In [5]:
import numpy as np
from scipy.special import gamma
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Generate a Mock Dataset (similar to the paper's setup, e.g., Diabetes or Heart)
X, y = make_classification(n_samples=200, n_features=20, n_informative=8, random_state=42)
X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Dataset generated with {X.shape[0]} samples and {X.shape[1]} features.")

Dataset generated with 200 samples and 20 features.


In [6]:
class BinaryCuckooSearch:
    def __init__(self, n_nests=10, max_iter=50, pa=0.35, alpha=1.0, beta=1.5):
        self.n_nests = n_nests
        self.max_iter = max_iter
        self.pa = pa  # Discovery rate of alien eggs/nests
        self.alpha = alpha  # Step size parameter
        self.beta = beta  # Levy distribution parameter

    def _levy_flight(self, Lambda):
        # Mantegna algorithm to generate Levy flights steps
        sigma_u = (gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
                  (gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2)))**(1 / Lambda)
        u = np.random.normal(0, sigma_u, size=self.dim)
        v = np.random.normal(0, 1, size=self.dim)
        step = u / (np.abs(v)**(1 / Lambda))
        return step

    def _sigmoid(self, x):
        # Clip input values to prevent overflow warning in exp
        clipped_x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-clipped_x))

    def fitness(self, nest_binary, X_tr, X_ev, y_tr, y_ev):
        selected_features = np.where(nest_binary == 1)[0]
        if len(selected_features) == 0:
            return 0.0  # Cannot evaluate with 0 features

        # Using Decision Tree Classifier as a fast surrogate for OPF classifier
        clf = DecisionTreeClassifier(random_state=42)
        clf.fit(X_tr[:, selected_features], y_tr)
        preds = clf.predict(X_ev[:, selected_features])
        return accuracy_score(y_ev, preds)

    def fit(self, X_tr, X_ev, y_tr, y_ev):
        self.dim = X_tr.shape[1]

        # Initialize continuous positions (X) and binary positions (Nest)
        X_continuous = np.random.uniform(-3, 3, (self.n_nests, self.dim))
        Nests = np.zeros((self.n_nests, self.dim), dtype=int)

        # Sigmoid transfer to binary
        for i in range(self.n_nests):
            sig = self._sigmoid(X_continuous[i])
            Nests[i] = np.where(sig > np.random.rand(self.dim), 1, 0)

        # Initial fitness evaluation
        fitness_values = np.array([self.fitness(nest, X_tr, X_ev, y_tr, y_ev) for nest in Nests])

        best_idx = np.argmax(fitness_values)
        best_nest = Nests[best_idx].copy()
        best_fitness = fitness_values[best_idx]

        for t in range(self.max_iter):
            # Generation of new solutions (Levy flights)
            for i in range(self.n_nests):
                step = self._levy_flight(self.beta)
                # Update continuous space
                X_continuous[i] = X_continuous[i] + self.alpha * step * (X_continuous[i] - best_nest)

                # Map to binary space using Transfer Function
                sig_val = self._sigmoid(X_continuous[i])
                Nests[i] = np.where(sig_val > np.random.rand(self.dim), 1, 0)

            # Evaluate fitness of newly created nests
            fitness_values = np.array([self.fitness(nest, X_tr, X_ev, y_tr, y_ev) for nest in Nests])

            # Discovery of alien eggs with probability Pa
            for i in range(self.n_nests):
                if np.random.rand() < self.pa:
                    # Build a new nest (abandoning the old worst ones)
                    X_continuous[i] = np.random.uniform(-3, 3, self.dim)
                    sig_val = self._sigmoid(X_continuous[i])
                    Nests[i] = np.where(sig_val > np.random.rand(self.dim), 1, 0)
                    fitness_values[i] = self.fitness(Nests[i], X_tr, X_ev, y_tr, y_ev)

            # Find the global best nest
            current_best_idx = np.argmax(fitness_values)
            if fitness_values[current_best_idx] > best_fitness:
                best_fitness = fitness_values[current_best_idx]
                best_nest = Nests[current_best_idx].copy()

            print(f"Iteration {t+1}/{self.max_iter} - Best Fitness (Accuracy): {best_fitness:.4f} | Features selected: {np.sum(best_nest)}")

        return best_nest, best_fitness

In [7]:
# Run the BCS Feature Selection optimizer
bcs = BinaryCuckooSearch(n_nests=15, max_iter=20, pa=0.35)
best_features, best_acc = bcs.fit(X_train, X_eval, y_train, y_eval)

selected_indices = np.where(best_features == 1)[0]
print(f"\nOptimal Feature Mask: {best_features}")
print(f"Selected Features indices: {selected_indices}")
print(f"Best Accuracy achieved: {best_acc * 100:.2f}%")

Iteration 1/20 - Best Fitness (Accuracy): 0.7000 | Features selected: 8
Iteration 2/20 - Best Fitness (Accuracy): 0.7167 | Features selected: 11
Iteration 3/20 - Best Fitness (Accuracy): 0.7167 | Features selected: 11
Iteration 4/20 - Best Fitness (Accuracy): 0.7333 | Features selected: 8
Iteration 5/20 - Best Fitness (Accuracy): 0.7333 | Features selected: 8
Iteration 6/20 - Best Fitness (Accuracy): 0.7333 | Features selected: 8
Iteration 7/20 - Best Fitness (Accuracy): 0.7333 | Features selected: 8
Iteration 8/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 9/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 10/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 11/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 12/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 13/20 - Best Fitness (Accuracy): 0.7500 | Features selected: 10
Iteration 14/20 - Best Fitness (Accuracy): 0.7500 | 